<a href="https://colab.research.google.com/github/javageek2018/AirlineArrivalDelay/blob/%E2%80%9Cflight_data%E2%80%9D/Flights_All_Features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [ ]:
from pyspark.sql.functions import col
import pyspark
import os
from pyspark.sql import SparkSession
from pyspark.sql import Window
from pyspark.sql import functions as F
import shutil, os, glob

# Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Set Up Spark

In [ ]:
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [ ]:
spark = SparkSession.builder \
    .appName("ColabSpark") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

spark

# Read Data (Spark)

In [ ]:
spark.conf.set("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED")
spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")

In [ ]:
# train_file_path = "/content/drive/MyDrive/OMDS Capstone/Data/flights_split/train_2018_2022.parquet"
# validate_file_path = "/content/drive/MyDrive/OMDS Capstone/Data/flights_split/validate_2023.parquet"
# test_file_path = "/content/drive/MyDrive/OMDS Capstone/Data/flights_split/test_2024.parquet"

train_file_path = "/content/drive/MyDrive/OMDS Capstone/Data/flights_encoded/train_encoded.parquet"
validate_file_path = "/content/drive/MyDrive/OMDS Capstone/Data/flights_encoded/validate_encoded.parquet"
test_file_path = "/content/drive/MyDrive/OMDS Capstone/Data/flights_encoded/test_encoded.parquet"


df_train = spark.read.parquet(train_file_path)
df_validate = spark.read.parquet(validate_file_path)
df_test = spark.read.parquet(test_file_path)

In [ ]:
print(f"Train: {df_train.count():,} rows")
print(f"Val  : {df_validate.count():,} rows")
print(f"Test : {df_test.count():,} rows")

Train: 31,149,502 rows
Val  : 6,743,403 rows
Test : 6,965,246 rows


In [ ]:
df_train.head()

Row(Year=2018, Quarter=1, Month=1, DayofMonth=1, DayOfWeek=1, FlightDate=1514764800000000000, Reporting_Airline='WN', Flight_Number_Reporting_Airline='1491.0', Origin='ABQ', Dest='BWI', CRSDepTime=730, DepTimeBlk='0700-0759', CRSArrTime=1310, ArrDel15=0, CRSElapsedTime=220.0, Distance=1670.0, DistanceGroup=7, date='2018-01-01', dep_hour=7, arr_hour=13, dep_hour_minus2=5, arr_hour_minus2=11, origin_temp_f=21.0, origin_dewpoint_f=8.1, origin_humidity=56.88, origin_feels_like_f=9.17, origin_wind_kts=8.818181818181818, origin_gust_kts=0.0, origin_visibility=10.0, origin_precip_in=0.0, origin_wx_codes='none', origin_is_rain=0, origin_is_snow=0, origin_is_fog=0, origin_low_visibility=0, origin_high_wind=0, origin_severe_weather=0, dest_temp_f=21.9, dest_dewpoint_f=3.0, dest_humidity=43.38, dest_feels_like_f=9.7, dest_wind_kts=9.692307692307692, dest_gust_kts=16.0, dest_visibility=10.0, dest_precip_in=0.0, dest_wx_codes='none', dest_is_rain=0, dest_is_snow=0, dest_is_fog=0, dest_low_visibilit

In [ ]:
df_train.dtypes

[('Year', 'bigint'),
 ('Quarter', 'bigint'),
 ('Month', 'bigint'),
 ('DayofMonth', 'bigint'),
 ('DayOfWeek', 'bigint'),
 ('FlightDate', 'bigint'),
 ('Reporting_Airline', 'string'),
 ('Flight_Number_Reporting_Airline', 'string'),
 ('Origin', 'string'),
 ('Dest', 'string'),
 ('CRSDepTime', 'bigint'),
 ('DepTimeBlk', 'string'),
 ('CRSArrTime', 'bigint'),
 ('ArrDel15', 'bigint'),
 ('CRSElapsedTime', 'double'),
 ('Distance', 'double'),
 ('DistanceGroup', 'bigint'),
 ('date', 'string'),
 ('dep_hour', 'bigint'),
 ('arr_hour', 'bigint'),
 ('dep_hour_minus2', 'bigint'),
 ('arr_hour_minus2', 'bigint'),
 ('origin_temp_f', 'double'),
 ('origin_dewpoint_f', 'double'),
 ('origin_humidity', 'double'),
 ('origin_feels_like_f', 'double'),
 ('origin_wind_kts', 'double'),
 ('origin_gust_kts', 'double'),
 ('origin_visibility', 'double'),
 ('origin_precip_in', 'double'),
 ('origin_wx_codes', 'string'),
 ('origin_is_rain', 'int'),
 ('origin_is_snow', 'int'),
 ('origin_is_fog', 'int'),
 ('origin_low_visibili

# Add rolling features

  ✅ Compute global_delay_rate from TRAIN ONLY

  ✅ Union all three, compute rolling windows (backward-looking = safe)
  
  ✅ Fill NULLs with training-derived defaults
  
  ✅ Split back apart

## Compute cold start values from train only

In [ ]:
# Save the training defaults

# ── Capture BEFORE transforming df_train ──────────────────────────────────────
# These are the "cold start" fill values derived purely from training data
global_delay_rate = df_train.agg(F.avg("ArrDel15")).collect()[0][0]
print(f"Global training delay rate (cold-start fill): {global_delay_rate:.4f}")

Global training delay rate (cold-start fill): 0.1785


## Add to validation and test sets

In [ ]:
# Tag both data frames and combine

# Tag each split so we can separate them later
df_train_tagged = df_train.withColumn("_split", F.lit("train"))
df_val_tagged   = df_validate.withColumn("_split",   F.lit("val"))
df_test_tagged  = df_test.withColumn("_split",  F.lit("test"))

# Union — all must have the same schema (ArrDel15 may be unknown but should exist)
df_all = (
    df_train_tagged
    .unionByName(df_val_tagged)
    .unionByName(df_test_tagged)
)

In [ ]:
# Build shared timestamp column

df_all = df_all.withColumn(
    "flight_date_unix",
    (F.col("FlightDate") / 1_000_000_000).cast("long")
)

df_all = df_all.withColumn(
    "crs_dep_unix",
    F.col("flight_date_unix")
    + (F.col("CRSDepTime") / 100).cast("int") * 3600
    + (F.col("CRSDepTime") % 100) * 60
)

In [ ]:
# Build all backward-looking windows

SECS_PER_DAY = 86_400
SECS_3H      = 3 * 3600

# Carrier delay rates
w_carrier_30d = (Window.partitionBy("Reporting_Airline")
                       .orderBy("flight_date_unix")
                       .rangeBetween(-30 * SECS_PER_DAY, -1))

w_carrier_90d = (Window.partitionBy("Reporting_Airline")
                       .orderBy("flight_date_unix")
                       .rangeBetween(-90 * SECS_PER_DAY, -1))

# Origin/Dest delay rates (target encoding)
w_origin_30d = (Window.partitionBy("Origin")
                      .orderBy("flight_date_unix")
                      .rangeBetween(-30 * SECS_PER_DAY, -1))

w_origin_90d = (Window.partitionBy("Origin")
                      .orderBy("flight_date_unix")
                      .rangeBetween(-90 * SECS_PER_DAY, -1))

w_dest_30d = (Window.partitionBy("Dest")
                    .orderBy("flight_date_unix")
                    .rangeBetween(-30 * SECS_PER_DAY, -1))

w_dest_90d = (Window.partitionBy("Dest")
                    .orderBy("flight_date_unix")
                    .rangeBetween(-90 * SECS_PER_DAY, -1))

# Congestion (3-hour window over scheduled departure timestamp)
w_congestion = (Window.partitionBy("Origin")
                      .orderBy("crs_dep_unix")
                      .rangeBetween(-SECS_3H, -1))


In [ ]:
# Compute all rolling features

df_all = (
    df_all
    .withColumn("carrier_delay_rate_30d", F.avg("ArrDel15").over(w_carrier_30d))
    .withColumn("carrier_delay_rate_90d", F.avg("ArrDel15").over(w_carrier_90d))
    .withColumn("origin_delay_rate_30d",  F.avg("ArrDel15").over(w_origin_30d))
    .withColumn("origin_delay_rate_90d",  F.avg("ArrDel15").over(w_origin_90d))
    .withColumn("dest_delay_rate_30d",    F.avg("ArrDel15").over(w_dest_30d))
    .withColumn("dest_delay_rate_90d",    F.avg("ArrDel15").over(w_dest_90d))
    .withColumn("origin_departures_3h",   F.count("*").over(w_congestion))
)

In [ ]:
# Fill nulls with training-derived defaults

df_all = df_all.fillna({
    "carrier_delay_rate_30d": global_delay_rate,
    "carrier_delay_rate_90d": global_delay_rate,
    "origin_delay_rate_30d":  global_delay_rate,
    "origin_delay_rate_90d":  global_delay_rate,
    "dest_delay_rate_30d":    global_delay_rate,
    "dest_delay_rate_90d":    global_delay_rate,
    "origin_departures_3h":   0,
})

In [ ]:
# Split back apart and drop construction columns

DROP_COLS = ["flight_date_unix", "crs_dep_unix", "_split"]

df_train = df_all.filter(F.col("_split") == "train").drop(*DROP_COLS)
df_val   = df_all.filter(F.col("_split") == "val").drop(*DROP_COLS)
df_test  = df_all.filter(F.col("_split") == "test").drop(*DROP_COLS)

In [ ]:
print(f"Train: {df_train.count():,} rows")
print(f"Val  : {df_val.count():,} rows")
print(f"Test : {df_test.count():,} rows")

Train: 31,149,502 rows
Val  : 6,743,403 rows
Test : 6,965,246 rows


## Output parquet files with all features

In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/OMDS Capstone/Data/flights_all_features/"

train_path = f"{OUTPUT_DIR}/train.parquet"
val_path   = f"{OUTPUT_DIR}/val.parquet"
test_path  = f"{OUTPUT_DIR}/test.parquet"

In [ ]:
def write_single_parquet(df, output_dir, filename):
    """Write a Spark DataFrame to a single .parquet file."""
    tmp_dir   = os.path.join(output_dir, f"_tmp_{filename}")
    final_path = os.path.join(output_dir, filename)

    # 1. Write as a single-partition folder
    (df.coalesce(1)
       .write
       .mode("overwrite")
       .option("compression", "snappy")
       .parquet(tmp_dir))

    # 2. Find the lone part-*.parquet file
    part_file = glob.glob(os.path.join(tmp_dir, "part-*.parquet"))[0]

    # 3. Move it to the final location
    if os.path.exists(final_path):
        os.remove(final_path)
    shutil.move(part_file, final_path)

    # 4. Clean up the temp folder
    shutil.rmtree(tmp_dir)

    print(f"✅ {filename}: {df.count():,} rows → {final_path}")

In [ ]:

# Write all three splits
write_single_parquet(df_train, OUTPUT_DIR, "train.parquet")
write_single_parquet(df_val,   OUTPUT_DIR, "val.parquet")
write_single_parquet(df_test,  OUTPUT_DIR, "test.parquet")

✅ train.parquet: 31,149,502 rows → /content/drive/MyDrive/OMDS Capstone/Data/flights_all_features/train.parquet
✅ val.parquet: 6,743,403 rows → /content/drive/MyDrive/OMDS Capstone/Data/flights_all_features/val.parquet
✅ test.parquet: 6,965,246 rows → /content/drive/MyDrive/OMDS Capstone/Data/flights_all_features/test.parquet
